In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece ipywidgets psutil

In [ ]:
print("""
Models Used:
1. HuggingFaceTB/SmolLM2-360M
2. TinyLlama/TinyLlama-1.1B-Chat-v1.0
""")



Models Used:
1. HuggingFaceTB/SmolLM2-360M
2. TinyLlama/TinyLlama-1.1B-Chat-v1.0



In [ ]:
# ============================================================
# Import Libraries
# ============================================================

import torch
import time
import psutil
import re
import gc

from transformers import AutoTokenizer, AutoModelForCausalLM

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import pandas as pd

In [ ]:
# ============================================================
# GPU Setup
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

if device == "cuda":
    print("GPU Name:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )
else:
    print("Running on CPU")

Using device: cuda
GPU Name: Tesla T4
GPU Memory: 15.64 GB


In [ ]:
# ============================================================
# Load SmolLM2 Model
# ============================================================

SMOL_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

print("Loading SmolLM2 Model...")

smol_tokenizer = AutoTokenizer.from_pretrained(
    SMOL_MODEL_NAME
)

smol_model = AutoModelForCausalLM.from_pretrained(
    SMOL_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print("SmolLM2 Loaded Successfully")

Loading SmolLM2 Model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

SmolLM2 Loaded Successfully


In [ ]:
# ============================================================
# Load TinyLlama Model
# ============================================================

TINY_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading TinyLlama Model...")

tiny_tokenizer = AutoTokenizer.from_pretrained(
    TINY_MODEL_NAME
)

tiny_model = AutoModelForCausalLM.from_pretrained(
    TINY_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print("TinyLlama Loaded Successfully")

Loading TinyLlama Model...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama Loaded Successfully


In [ ]:
# ============================================================
# System Prompt
# ============================================================

SYSTEM_PROMPT = """
You are a helpful AI assistant.

Rules:
- Answer clearly and concisely.
- Do not generate fake User messages.
- Do not continue conversations unnecessarily.
- Stay relevant to the question.
- Avoid hallucinations.
- Keep responses informative.
"""

In [ ]:
# ============================================================
# Conversation Memory System
# ============================================================

conversation_history = []

MAX_HISTORY = 5


def build_prompt(user_message):

    global conversation_history

    prompt = SYSTEM_PROMPT + "\n\n"

    for user_text, assistant_text in conversation_history[-MAX_HISTORY:]:

        prompt += f"User: {user_text}\n"
        prompt += f"Assistant: {assistant_text}\n"

    prompt += f"User: {user_message}\nAssistant:"

    return prompt

In [ ]:
# ============================================================
# Response Cleaning Logic
# ============================================================

def clean_response(response):

    stop_patterns = [
        "User:",
        "Assistant:",
        "\nUser",
        "\nAssistant"
    ]

    for pattern in stop_patterns:

        if pattern in response:
            response = response.split(pattern)[0]

    response = re.sub(r"(Assistant:)+", "", response)

    response = response.strip()

    return response


In [ ]:
# ============================================================
# Chatbot Generation Function
# ============================================================

def generate_response(
    model_name,
    user_message,
    temperature=0.7,
    max_new_tokens=128,
    top_p=0.9,
    repetition_penalty=1.1
):

    prompt = build_prompt(user_message)

    if model_name == "SmolLM2":
        tokenizer = smol_tokenizer
        model = smol_model

    else:
        tokenizer = tiny_tokenizer
        model = tiny_model

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    start_time = time.time()

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generation_time = round(
        time.time() - start_time,
        2
    )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = decoded[len(prompt):]

    response = clean_response(response)

    memory_usage = round(
        psutil.Process().memory_info().rss / (1024 ** 2),
        2
    )

    conversation_history.append(
        (user_message, response)
    )

    return {
        "response": response,
        "time": generation_time,
        "memory": memory_usage
    }

In [ ]:
# ============================================================
# Interactive Chat Interface
# ============================================================

chat_output = widgets.Output(
    layout={
        "border": "1px solid #444",
        "height": "500px",
        "overflow_y": "auto"
    }
)

user_input = widgets.Text(
    placeholder="Type your message...",
    layout=widgets.Layout(width="70%")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary"
)

clear_button = widgets.Button(
    description="Clear Chat",
    button_style="danger"
)

model_dropdown = widgets.Dropdown(
    options=["SmolLM2", "TinyLlama"],
    value="TinyLlama",
    description="Model:"
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.5,
    step=0.1,
    description="Temperature"
)

max_tokens_slider = widgets.IntSlider(
    value=128,
    min=32,
    max=512,
    step=32,
    description="Max Tokens"
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.05,
    description="Top P"
)

repetition_slider = widgets.FloatSlider(
    value=1.1,
    min=1.0,
    max=2.0,
    step=0.1,
    description="Repeat Penalty"
)

loading_label = widgets.HTML(value="")


# ============================================================
# Message Display Function
# ============================================================

def add_message(sender, text):

    with chat_output:

        bubble_class = (
            "user-bubble"
            if sender == "User"
            else "assistant-bubble"
        )

        display(HTML(f"""
        <div class="{bubble_class}">
            <b>{sender}:</b><br>
            {text}
        </div>
        """))


# ============================================================
# Send Button Logic
# ============================================================

def on_send_clicked(b):

    message = user_input.value.strip()

    if not message:
        return

    add_message("User", message)

    user_input.value = ""

    loading_label.value = """
    <b style="color:orange;">
    Generating response...
    </b>
    """

    try:

        selected_model = model_dropdown.value

        result = generate_response(
            model_name=selected_model,
            user_message=message,
            temperature=temperature_slider.value,
            max_new_tokens=max_tokens_slider.value,
            top_p=top_p_slider.value,
            repetition_penalty=repetition_slider.value
        )

        response_text = result["response"]

        add_message(
            selected_model,
            response_text
        )

        add_message(
            "System",
            f"""
            Response Time: {result['time']} sec
            <br>
            Memory Usage: {result['memory']} MB
            """
        )

    except Exception as e:

        add_message(
            "Error",
            str(e)
        )

    loading_label.value = ""


# ============================================================
# Clear Chat Function
# ============================================================

def clear_chat(b):

    global conversation_history

    conversation_history = []

    chat_output.clear_output()


# ============================================================
# Button Events
# ============================================================

send_button.on_click(on_send_clicked)

clear_button.on_click(clear_chat)


# ============================================================
# Layout
# ============================================================

controls = widgets.VBox([
    model_dropdown,
    temperature_slider,
    max_tokens_slider,
    top_p_slider,
    repetition_slider
])

input_row = widgets.HBox([
    user_input,
    send_button,
    clear_button
])

ui = widgets.VBox([
    widgets.HTML(
        '<h2 style="color:white;">Mini AI Chatbot</h2>'
    ),
    controls,
    chat_output,
    loading_label,
    input_row
])

display(ui)

In [ ]:
benchmark_prompts = [
    "I love Machine learning.",
    "What is Natural Language Processing?",
]

benchmark_results = []

for prompt in benchmark_prompts:

    model_outputs = {}

    for model_name in ["SmolLM2", "TinyLlama"]:

        result = generate_response(
            model_name=model_name,
            user_message=prompt
        )

        response = result["response"]

        word_count = len(response.split())

        unique_words = len(set(response.split()))

        repetition_ratio = unique_words / word_count if word_count > 0 else 0

        quality_score = round(
            (word_count * 0.6) + (repetition_ratio * 40),
            2
        )

        benchmark_results.append({
            "Prompt": prompt,
            "Model": model_name,
            "Response Time (sec)": result["time"],
            "Quality Score": quality_score
        })

benchmark_df = pd.DataFrame(benchmark_results)

display(benchmark_df)

summary_df = benchmark_df.groupby("Model").agg({
    "Response Time (sec)": "mean",
    "Quality Score": "mean"
}).reset_index()

display(summary_df)

,Prompt,Model,Response Time (sec),Quality Score
0,I love Machine learning.,SmolLM2,5.84,65.49
1,I love Machine learning.,TinyLlama,4.31,56.16
2,What is Natural Language Processing?,SmolLM2,6.27,70.09
3,What is Natural Language Processing?,TinyLlama,4.14,63.67


,Model,Response Time (sec),Quality Score
0,SmolLM2,6.055,67.790
1,TinyLlama,4.225,59.915
